# Part 1 — Step 3: Faster R-CNN Training

**Why Faster R-CNN?**  
Faster R-CNN (Ren et al., NeurIPS 2015) is the canonical two-stage detector and a strong classic baseline.
Its Region Proposal Network (RPN) proposes candidate regions, which are then classified and refined.
The ResNet-50 + FPN backbone provides multi-scale features — important for detecting small acne lesions
at varying scales across a face image.

**This notebook:**
1. Loads ACNE04 COCO annotations directly (no format conversion needed)
2. Fine-tunes a pretrained Faster R-CNN (ResNet-50 + FPN) for 20 epochs
3. Saves the best checkpoint based on validation loss

**Output:** `outputs/faster_rcnn/best.pth`

In [ ]:
import json
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
DATA_DIR     = Path("../data/acne04")
OUT_DIR      = Path("../outputs/faster_rcnn")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS       = 20
BATCH        = 4
LR           = 1e-3
MOMENTUM     = 0.9
WEIGHT_DECAY = 1e-4
LR_STEP      = 5
LR_GAMMA     = 0.5
NUM_CLASSES  = 5    # background + 4 acne classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dataset

In [ ]:
class Acne04Dataset(Dataset):
    """
    Loads ACNE04 images and COCO bounding box annotations.
    Returns (image_tensor, target_dict) where target_dict has
    'boxes' [N,4] in x1y1x2y2 format and 'labels' [N] 1-indexed.
    """
    def __init__(self, split):
        self.img_dir = DATA_DIR / split
        with open(self.img_dir / "_annotations.coco.json") as f:
            coco = json.load(f)
        cats = sorted(coco["categories"], key=lambda c: c["id"])
        self.class_names  = ["__background__"] + [c["name"] for c in cats]
        self.cat_to_label = {c["id"]: i+1 for i, c in enumerate(cats)}
        self.images = {img["id"]: img for img in coco["images"]}
        self.ann_map = {}
        for ann in coco.get("annotations", []):
            self.ann_map.setdefault(ann["image_id"], []).append(ann)
        self.ids = list(self.images.keys())
        self.transform = transforms.ToTensor()

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        meta   = self.images[img_id]
        img    = Image.open(self.img_dir / meta["file_name"]).convert("RGB")
        anns   = self.ann_map.get(img_id, [])
        if anns:
            boxes  = torch.tensor([[a["bbox"][0], a["bbox"][1],
                                    a["bbox"][0]+a["bbox"][2],
                                    a["bbox"][1]+a["bbox"][3]] for a in anns], dtype=torch.float32)
            labels = torch.tensor([self.cat_to_label[a["category_id"]] for a in anns], dtype=torch.int64)
        else:
            boxes  = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
        return self.transform(img), {"boxes": boxes, "labels": labels,
                                     "image_id": torch.tensor([img_id])}

def collate_fn(batch): return tuple(zip(*batch))

train_ds = Acne04Dataset("train")
val_ds   = Acne04Dataset("valid")
print(f"Train: {len(train_ds)} images  |  Val: {len(val_ds)} images")
print(f"Classes: {train_ds.class_names}")

## 2. Build model

In [ ]:
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(device)
print("Model ready.")

## 3. Training loop

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, collate_fn=collate_fn, num_workers=2)

optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

train_losses, val_losses = [], []
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    total = 0.0
    for imgs, targets in train_loader:
        imgs    = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss = sum(model(imgs, targets).values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item()
    scheduler.step()
    avg_train = total / len(train_loader)
    train_losses.append(avg_train)

    # Validate
    model.train()   # keep train mode to get losses
    total = 0.0
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs    = [i.to(device) for i in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            total  += sum(model(imgs, targets).values()).item()
    avg_val = total / len(val_loader)
    val_losses.append(avg_val)

    print(f"Epoch [{epoch:02d}/{EPOCHS}]  train={avg_train:.4f}  val={avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), OUT_DIR / "best.pth")
        print(f"  ✓ Best checkpoint saved (val={avg_val:.4f})")

torch.save(model.state_dict(), OUT_DIR / "last.pth")
print(f"\nDone. Best val loss: {best_val_loss:.4f}")

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses,   label="Val loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("Faster R-CNN — Training & Validation Loss")
plt.legend(); plt.tight_layout()
plt.savefig("../outputs/figures/faster_rcnn_loss.png", dpi=150)
plt.show()